# Mini Project 5 - Analisis Spotify Charts dengan Apache Spark

Notebook ini mengerjakan tahapan: pemahaman data, visualisasi popularitas artis, streaming menurut wilayah, popularitas lagu di Spotify, heatmap korelasi antar variabel, lagu paling populer dari setiap negara, dan top 100 rank setiap negara berdasarkan bulan tertentu.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


## 1. Setup Spark dan Load Dataset

Dataset cukup besar, jadi pembacaan dilakukan dengan Apache Spark dan schema eksplisit agar tidak perlu infer schema ke seluruh file.


In [ ]:
spark = (
    SparkSession.builder
    .appName("MiniProject5SpotifyCharts")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("title", StringType(), True),
    StructField("rank", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("artist", StringType(), True),
    StructField("url", StringType(), True),
    StructField("region", StringType(), True),
    StructField("chart", StringType(), True),
    StructField("trend", StringType(), True),
    StructField("streams", LongType(), True),
])

file_path = "charts.csv"

df = (
    spark.read
    .option("header", True)
    .option("multiLine", False)
    .option("escape", '"')
    .option("dateFormat", "yyyy-MM-dd")
    .schema(schema)
    .csv(file_path)
)

df = df.withColumn("month", date_format(col("date"), "yyyy-MM"))
df.cache()


## 2. Pemahaman Data

Bagian ini menampilkan schema, contoh data, jumlah baris, rentang tanggal, jumlah wilayah, jenis chart, dan missing value.


In [ ]:
df.printSchema()
print(f"Jumlah baris: {df.count():,}")
print(f"Jumlah kolom: {len(df.columns)}")

df.show(10, truncate=False)


In [ ]:
summary_df = df.agg(
    min("date").alias("tanggal_awal"),
    max("date").alias("tanggal_akhir"),
    count_distinct("region").alias("jumlah_negara_wilayah"),
    count_distinct("title").alias("jumlah_lagu"),
    count_distinct("artist").alias("jumlah_artis"),
    sum("streams").alias("total_streams")
)
summary_df.show(truncate=False)

print("Jenis chart:")
df.groupBy("chart").count().orderBy(desc("count")).show(truncate=False)

print("Trend:")
df.groupBy("trend").count().orderBy(desc("count")).show(truncate=False)


In [ ]:
missing_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])
missing_df.show(truncate=False)


## 3. Visualisasi Popularitas Artis

Popularitas artis dihitung dari total streams, jumlah kemunculan di chart, rata-rata rank, dan rank terbaik.


In [ ]:
artist_popularity = (
    df.groupBy("artist")
    .agg(
        sum("streams").alias("total_streams"),
        count("*").alias("jumlah_masuk_chart"),
        avg("rank").alias("rata_rata_rank"),
        min("rank").alias("rank_terbaik"),
        count_distinct("title").alias("jumlah_lagu")
    )
    .orderBy(desc("total_streams"))
)

artist_popularity.show(20, truncate=False)

top_artist_pd = artist_popularity.limit(20).toPandas()

plt.figure(figsize=(12, 7))
sns.barplot(data=top_artist_pd, y="artist", x="total_streams", palette="viridis")
plt.title("Top 20 Artis Berdasarkan Total Streams")
plt.xlabel("Total Streams")
plt.ylabel("Artis")
plt.tight_layout()
plt.show()


## 4. Streaming Menurut Wilayah

Agregasi total streams per negara/wilayah untuk melihat pasar dengan streaming terbesar.


In [ ]:
region_streaming = (
    df.groupBy("region")
    .agg(
        sum("streams").alias("total_streams"),
        avg("streams").alias("rata_rata_streams"),
        count("*").alias("jumlah_data_chart"),
        count_distinct("title").alias("jumlah_lagu_unik")
    )
    .orderBy(desc("total_streams"))
)

region_streaming.show(30, truncate=False)

top_region_pd = region_streaming.limit(20).toPandas()

plt.figure(figsize=(12, 7))
sns.barplot(data=top_region_pd, y="region", x="total_streams", palette="mako")
plt.title("Top 20 Negara/Wilayah Berdasarkan Total Streams")
plt.xlabel("Total Streams")
plt.ylabel("Negara/Wilayah")
plt.tight_layout()
plt.show()


## 5. Popularitas Lagu di Spotify

Popularitas lagu dihitung dari total streams, jumlah hari masuk chart, banyak wilayah, rata-rata rank, dan peak rank.


In [ ]:
song_popularity = (
    df.groupBy("title", "artist")
    .agg(
        sum("streams").alias("total_streams"),
        count("*").alias("jumlah_masuk_chart"),
        count_distinct("region").alias("jumlah_wilayah"),
        count_distinct("date").alias("jumlah_hari"),
        avg("rank").alias("rata_rata_rank"),
        min("rank").alias("rank_terbaik")
    )
    .orderBy(desc("total_streams"))
)

song_popularity.show(20, truncate=False)

top_song_pd = song_popularity.limit(20).toPandas()
top_song_pd["song_artist"] = top_song_pd["title"] + " - " + top_song_pd["artist"]

plt.figure(figsize=(12, 8))
sns.barplot(data=top_song_pd, y="song_artist", x="total_streams", palette="crest")
plt.title("Top 20 Lagu Spotify Berdasarkan Total Streams")
plt.xlabel("Total Streams")
plt.ylabel("Lagu - Artis")
plt.tight_layout()
plt.show()


## 6. Heatmap Korelasi Antar Variabel

Variabel kategorikal sederhana seperti panjang judul, panjang nama artis, dan indikator trend diubah menjadi angka agar bisa dikorelasikan dengan rank dan streams.


In [ ]:
correlation_df = df.select(
    col("rank").cast("double"),
    col("streams").cast("double"),
    length("title").cast("double").alias("title_length"),
    length("artist").cast("double").alias("artist_length"),
    when(col("trend") == "MOVE_UP", 1.0).otherwise(0.0).alias("move_up"),
    when(col("trend") == "MOVE_DOWN", 1.0).otherwise(0.0).alias("move_down"),
    when(col("trend") == "NEW_ENTRY", 1.0).otherwise(0.0).alias("new_entry")
).dropna()

sample_corr_pd = correlation_df.sample(False, 0.01, seed=42).limit(200000).toPandas()
corr_matrix = sample_corr_pd.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Heatmap Korelasi Antar Variabel")
plt.tight_layout()
plt.show()


## 7. Lagu Paling Populer dari Setiap Negara

Lagu paling populer setiap negara ditentukan berdasarkan total streams tertinggi pada masing-masing negara/wilayah.


In [ ]:
song_by_country = (
    df.groupBy("region", "title", "artist")
    .agg(
        sum("streams").alias("total_streams"),
        min("rank").alias("rank_terbaik"),
        avg("rank").alias("rata_rata_rank"),
        count_distinct("date").alias("jumlah_hari_masuk_chart")
    )
)

country_window = Window.partitionBy("region").orderBy(desc("total_streams"), asc("rank_terbaik"))

most_popular_song_each_country = (
    song_by_country
    .withColumn("nomor", row_number().over(country_window))
    .filter(col("nomor") == 1)
    .drop("nomor")
    .orderBy("region")
)

most_popular_song_each_country.show(200, truncate=False)


## 8. Top 100 Rank Setiap Negara Berdasarkan Bulan Tertentu

Ubah nilai `bulan_tertentu` sesuai bulan yang ingin dianalisis dengan format `YYYY-MM`. Query ini menampilkan top 100 rank untuk setiap negara pada bulan tersebut. Jika satu lagu muncul berkali-kali dalam bulan yang sama, data diringkas menggunakan rank terbaik dan total streams bulanan.


In [ ]:
bulan_tertentu = "2017-01"  # Ganti contoh: "2020-05", "2021-12", dst.

top_100_each_country_month = (
    df.filter(col("month") == bulan_tertentu)
    .groupBy("region", "title", "artist")
    .agg(
        min("rank").alias("rank_terbaik_bulan"),
        sum("streams").alias("total_streams_bulan"),
        count_distinct("date").alias("jumlah_hari_masuk_chart")
    )
)

monthly_window = Window.partitionBy("region").orderBy(asc("rank_terbaik_bulan"), desc("total_streams_bulan"))

top_100_each_country_month = (
    top_100_each_country_month
    .withColumn("rank_bulanan", row_number().over(monthly_window))
    .filter(col("rank_bulanan") <= 100)
    .orderBy("region", "rank_bulanan")
)

top_100_each_country_month.show(500, truncate=False)


## 9. Simpan Hasil Opsional

Jalankan cell ini jika ingin menyimpan hasil analisis ke folder output. Format Parquet lebih cocok untuk hasil Spark karena cepat dibaca kembali.


In [ ]:
# most_popular_song_each_country.write.mode("overwrite").parquet("output/most_popular_song_each_country")
# top_100_each_country_month.write.mode("overwrite").parquet(f"output/top_100_each_country_{bulan_tertentu}")
